SQL script for silver to gold transformation

In [ ]:
use database healthpulse_manual_db;

In [ ]:
-- Create gold schema
create schema if not exists gold_schema;

DIM_PATIENT TABLE

In [ ]:
-- check if there are multiple entries for same patient_id
SELECT 
    patient_id,
    COUNT(DISTINCT age) AS age_count,
    COUNT(DISTINCT insurance_type) AS insurance_count,
    
    LISTAGG(DISTINCT age, ', ') AS ages,
    LISTAGG(DISTINCT insurance_type, ', ') AS insurance_types
    
FROM silver_schema.silver_data
GROUP BY patient_id

HAVING COUNT(DISTINCT age) > 1
    OR COUNT(DISTINCT insurance_type) > 1;

In [ ]:
-- create and insert in gold schema - dim_patient table
CREATE OR REPLACE TABLE gold_schema.dim_patients (
    patient_id VARCHAR(20) PRIMARY KEY,
    age NUMBER(3),
    insurance_type VARCHAR(20)
);

INSERT INTO gold_schema.dim_patients (patient_id, age, insurance_type)
SELECT DISTINCT
    patient_id,
    age,
    insurance_type
FROM silver_schema.silver_data;

In [ ]:
select count(distinct patient_id) from silver_schema.silver_data;

DIM_PROVIDER_TABLE

In [ ]:
-- check if there are multiple entries for same provider_id
SELECT 
    provider_id,
    COUNT(DISTINCT specialty) AS specialty_count,
    COUNT(DISTINCT clinic_assignment) AS clinic_assignment_count,

    LISTAGG(DISTINCT specialty, ', ') AS specialties,
    LISTAGG(DISTINCT clinic_assignment, ', ') AS clinic_assisgnments
    
FROM silver_schema.silver_data
GROUP BY provider_id
HAVING COUNT(DISTINCT specialty) > 1
    Or COUNT(DISTINCT clinic_assignment) > 1;

In [ ]:
CREATE OR REPLACE TABLE gold_schema.dim_providers (
    provider_id VARCHAR(20) PRIMARY KEY,
    specialty VARCHAR(100),
    clinic_assignment VARCHAR(20)
);

INSERT INTO gold_schema.dim_providers (provider_id, specialty, clinic_assignment)
SELECT DISTINCT
    provider_id,
    specialty,
    clinic_assignment
FROM silver_schema.silver_data;

In [ ]:
select count(distinct provider_id) from silver_schema.silver_data;

DIM_CLINIC TABLE

In [ ]:
-- check if there are multiple entries for same provider_clinic_id
SELECT 
    provider_clinic_id,
    COUNT(DISTINCT clinic_name) AS cname_count,
    COUNT(DISTINCT city) AS city_count,
    COUNT(DISTINCT hours) AS hours_count,

    LISTAGG(DISTINCT clinic_name, ', ') AS clinics,
    LISTAGG(DISTINCT city, ', ') AS cites,
    LISTAGG(DISTINCT hours, ', ') AS hours
    
FROM silver_schema.silver_data
GROUP BY provider_clinic_id
HAVING COUNT(DISTINCT clinic_name) > 1
    Or COUNT(DISTINCT city) > 1
    Or COUNT(DISTINCT hours) > 1;

In [ ]:
CREATE OR REPLACE TABLE gold_schema.dim_clinics (
    provider_clinic_id VARCHAR(20) PRIMARY KEY,
    clinic_name VARCHAR(100),
    city VARCHAR(20),
    hours VARCHAR(20)
);

INSERT INTO gold_schema.dim_clinics (provider_clinic_id, clinic_name, city, hours)
SELECT DISTINCT
    provider_clinic_id,
    clinic_name,
    city,
    hours
FROM silver_schema.silver_data;

In [ ]:
select count(distinct provider_clinic_id) from silver_schema.silver_data;

DIM_DATE TABLE

In [ ]:
CREATE OR REPLACE TABLE gold_schema.dim_dates (
    date_id INT PRIMARY KEY,
    appointment_date DATE,
    year INT,
    month INT,
    day INT,
    day_of_week INT
);

INSERT INTO gold_schema.dim_dates (date_id, appointment_date, year, month, day, day_of_week)
SELECT
    ROW_NUMBER() OVER (ORDER BY appointment_date) AS date_id,
    appointment_date,
    YEAR(appointment_date) AS year,
    MONTH(appointment_date) AS month,
    DAY(appointment_date) AS day,
    DAYOFWEEK(appointment_date) AS day_of_week
FROM (
    SELECT DISTINCT appointment_date
    FROM silver_schema.silver_data
) AS unique_dates
ORDER BY appointment_date;

FACT_APPOINTMENT TABLE

In [ ]:
CREATE OR REPLACE TABLE gold_schema.fact_appointments (
    appointment_id VARCHAR(20) PRIMARY KEY,
    patient_id VARCHAR(20),
    provider_id VARCHAR(20),
    provider_clinic_id VARCHAR(20),
    date_id NUMBER(38,0),
    appointment_time TIME,
    lead_time_days NUMBER(5),
    wait_time_minutes NUMBER(5,2),
    is_no_show NUMBER(2),
    status VARCHAR(20),

    constraint fk_dim_patient foreign key (patient_id) references gold_schema.dim_patients (patient_id),
    constraint fk_dim_provider foreign key (provider_id) references gold_schema.dim_providers (provider_id),
    constraint fk_dim_clinic foreign key (provider_clinic_id) references gold_schema.dim_clinics (provider_clinic_id),
    constraint fk_dim_date foreign key (date_id) references gold_schema.dim_dates (date_id)
);

In [ ]:
INSERT INTO gold_schema.fact_appointments (
    appointment_id,
    patient_id,
    provider_id,
    provider_clinic_id,
    date_id,
    appointment_time,
    lead_time_days,
    wait_time_minutes,
    is_no_show,
    status
)
SELECT
    s.appointment_id,
    s.patient_id,
    s.provider_id,
    s.provider_clinic_id,
    dd.date_id,
    s.appointment_time,
    s.lead_time_days,
    s.wait_time_minutes,
    s.is_no_show,
    s.status
FROM silver_schema.silver_data as s
       
JOIN gold_schema.dim_dates as dd
    ON s.appointment_date = dd.appointment_date;


TEST CASE

In [ ]:
select * from gold_schema.fact_appointments as f 
join gold_schema.dim_patients as d1 on f.patient_id = d1.patient_id
join gold_schema.dim_providers as d2 on f.provider_id = d2.provider_id
join gold_schema.dim_clinics as d3 on f.provider_clinic_id = d3.provider_clinic_id
join gold_schema.dim_dates as d4 on f.date_id = d4.date_id
where f.appointment_id = 'A0100001'

In [ ]:
select * from silver_schema.silver_data where appointment_id = 'A0100001'